# Task 1 — Data understanding and EDA

This notebook runs the repository's bounded-memory Parquet audit. The official train parquet is about 3 GB; use a mounted Drive path rather than loading the entire table into RAM. Kaggle competition data require accepting the official rules. Test parquet alone is sufficient for test schema/spectrum EDA; train/test comparison needs train parquet.

In [ ]:
%pip -q install pyarrow pandas numpy matplotlib seaborn
from pathlib import Path
import sys

# If running in Colab, uncomment and mount Drive, then set DATA_DIR accordingly.
# from google.colab import drive
# drive.mount('/content/drive')
DATA_DIR = Path('/content/drive/MyDrive/casmi_data')  # change to the directory containing parquet files
OUT_DIR = Path('/content/casmi_task1_eda')

# Clone the current implementation branch (or set REPO to your existing checkout).
import subprocess
REPO = Path('/content/Enveda-CASMI-2026')
if not (REPO / 'src' / 'task1_eda.py').is_file():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'arena/00d6d90f-enveda-casmi-2026', 'https://github.com/KrishanChauhanAI/Enveda-CASMI-2026.git', str(REPO)], check=True)
sys.path.insert(0, str(REPO / 'src'))
from task1_eda import scan_parquet, plot_eda


In [ ]:
test_path = DATA_DIR / 'test.parquet'
train_path = DATA_DIR / 'train.parquet'
assert test_path.is_file(), f'Missing test parquet: {test_path}'
reports, samples = {}, {}
if train_path.is_file():
    reports['train'], samples['train'] = scan_parquet(train_path, sample_rows=100_000, batch_size=25_000)
else:
    print('Training file not found; running test-only EDA. No supervised training is performed in Task 1.')
reports['test'], samples['test'] = scan_parquet(test_path, sample_rows=100_000, batch_size=25_000)
OUT_DIR.mkdir(parents=True, exist_ok=True)
(OUT_DIR / 'data_dictionary_and_quality.json').write_text(__import__('json').dumps(reports, indent=2, default=str))
plot_eda(samples.get('train'), samples.get('test'), OUT_DIR / 'composite_eda.png')
print({k: {'rows': v['rows'], 'unique_molecules': v['unique_molecules'], 'columns': [c['name'] for c in v['columns']]} for k, v in reports.items()})


In [ ]:
from IPython.display import display, Image
display(Image(filename=str(OUT_DIR / 'composite_eda.png')))
display(reports)


## Acceptance checks

- The data dictionary reports actual Arrow column types and null counts for each file.
- `molecule_id` counts are distinct molecule counts while `rows` counts spectra; multiple spectra per molecule are expected.
- Train rows are scanned in batches; the plotting sample is capped, so the full train parquet is never loaded into one pandas DataFrame.
- Composite figures cover precursor mass, peak counts, collision energy and—when present—training library, adduct and instrument distributions.
- Save/inspect `data_dictionary_and_quality.json` and `composite_eda.png`; do not begin Task 2 until schema and data quality are reviewed.
